# 01 - Integrate Controls

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import rapids_singlecell as rsc
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
from dynaconf import Dynaconf

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data 

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_processed.h5ad")

In [ ]:
adata

In [ ]:
adata.obs["sample"].unique().tolist()

## Add plate assignment

In [ ]:
# load and clean plate assignments
plate_assignments = (
    pd.read_csv("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis/plate_assignment.csv")
    .rename(columns={
        "Sample Name": "sample",
        "TF": "condition",
        "Batch": "batch",
        "Round within Batch": "round_within_batch",
    })
)

# standardize sample IDs
plate_assignments["sample"] = (
    plate_assignments["sample"]
    .str.replace(".", "_")
    .apply(lambda x: f"sample_{x}" if not any(c.isalpha() for c in x) else x)
)

# merge with obs
metadata = (
    adata.obs.reset_index()
    .merge(plate_assignments, on="sample", how="left")
    .set_index("bc_wells")
)

# normalize condition labels
metadata["condition"] = metadata["condition"].str.lower().str.capitalize()

In [ ]:
plate_assignments[plate_assignments["condition"].isin(["Control"])]

In [ ]:
# update adata
adata.obs = metadata

## Subset to controls only

In [ ]:
adata = adata[adata.obs["condition"].isin(["Control"])]

In [ ]:
adata = adata.raw.to_adata()

In [ ]:
adata.layers['counts'] = adata.X
adata.raw = adata.copy()

## Perform doublet detection

In [ ]:
rsc.get.anndata_to_GPU(adata)

In [ ]:
rsc.pp.scrublet(adata, batch_key="sample")

In [ ]:
rsc.get.anndata_to_CPU(adata)

In [ ]:
# Library size normalization
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform counts
sc.pp.log1p(adata)

# Store the log-normalized data in a separate layer
adata.layers["lognorm"] = adata.X.copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    n_top_genes=5000,
    batch_key="sample"  
)

In [ ]:
adata = adata[:, adata.var["highly_variable"]].copy()

## Dimensionality reduction

In [ ]:
rsc.get.anndata_to_GPU(adata)

In [ ]:
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_mito"])

In [ ]:
rsc.pp.scale(adata, max_value=10)

In [ ]:
rsc.pp.pca(adata, n_comps=100, use_highly_variable=False)

In [ ]:
rsc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)
rsc.tl.umap(adata)

In [ ]:
for resolution in [0.3, 0.5, 1.0, 1.5, 2.0]:
    rsc.tl.leiden(adata, resolution=resolution, key_added=f"pca_leiden_{resolution}")  

In [ ]:
sc.pl.embedding(adata, basis="X_umap", color=["sample", "predicted_doublet", "pca_leiden_2.0"], wspace=0.4, frameon=False, legend_loc="on data", show=True)

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_combined_controls_processed.h5ad")

## Subclustering of I/K/D cells

In [ ]:
import anndata as ad

def subcluster(
    adata: ad.AnnData,
    cluster_key: str,
    clusters_to_sub: list,
    new_key: str = "subcluster",
    use_rep: str = "X_pca",
    resolution: float = 1.0,
):
    """
    Subcluster only selected clusters in an AnnData object.
    Keeps all other cluster labels unchanged.
    
    Parameters
    ----------
    adata : AnnData
        Input object with clustering in `.obs[cluster_key]`.
    cluster_key : str
        Column in .obs that stores the original clusters.
    clusters_to_sub : list
        Which cluster labels to subcluster (must match dtype in cluster_key).
    new_key : str, optional
        Name of new obs column to store results.
    use_rep : str, optional
        Which representation to use for neighbor graph construction.
    resolution : float, optional
        Resolution for Leiden clustering.
    """
    # start from original cluster labels
    adata.obs[new_key] = adata.obs[cluster_key].astype(str).copy()

    for c in clusters_to_sub:
        # subset AnnData to just one cluster
        subset = adata[adata.obs[cluster_key] == c].copy()

        # recompute neighbors + clustering
        rsc.pp.neighbors(subset, use_rep=use_rep)
        rsc.tl.leiden(subset, resolution=resolution, key_added="tmp_sub")

        # relabel: prefix with cluster name
        mapping = {
            cell: f"{c}_{sub}" for cell, sub in zip(subset.obs_names, subset.obs["tmp_sub"])
        }
        adata.obs.loc[subset.obs_names, new_key] = [
            mapping[cell] for cell in subset.obs_names
        ]

    return adata

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_controls_processed.h5ad")

In [ ]:
adata = subcluster(adata, cluster_key="pca_leiden_2.0", clusters_to_sub=["7", "15"], new_key="clust", resolution=0.5)

In [ ]:
sc.pl.embedding(adata, basis="X_umap", color=["sample", "clust"], wspace=0.4, frameon=False, legend_loc="on data", show=True)

In [ ]:
del adata.uns["log1p"]
del adata.uns["pca"]

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_combined_controls_processed.h5ad")

## Visualize

In [ ]:
adata = adata.raw.to_adata()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
del adata.uns["log1p"]

In [ ]:
adata

In [ ]:
del adata.uns["clust_colors"]
del adata.uns["pca_leiden_2.0_colors"]
del adata.uns["predicted_doublet_colors"]
del adata.uns["sample_colors"]

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_combined_controls_processed_vis.h5ad")